# NB 5 — Tool retrieval and model routing
**Goal:** two cheap moves that happen *before* the agent acts and quietly carry a lot of the value — **narrowing a big tool set to the few relevant tools**, and **routing each task to the cheapest model that can handle it.**

An agent with 50 tools and one expensive model is slow, costly, and error-prone. Retrieval shrinks the action space; routing (a cascade) spends the big model only where it's needed. (Runs in MOCK mode with no API key.)

In [1]:
import os, json, re

# =============================================================
# Model backend — works two ways:
#   1) MOCK (default): no API key needed. Returns scripted responses
#      so you can run the whole notebook and see the STRUCTURE.
#   2) REAL model: pip install openai, then either
#        - Cloud:  export OPENAI_API_KEY=sk-...        (uses OpenAI)
#        - Local open-weight (vLLM / LM Studio / Ollama):
#              export OPENAI_BASE_URL=http://localhost:8000/v1
#              export OPENAI_API_KEY=dummy
#              export MODEL=meta-llama/Llama-3.1-8B-Instruct   # your served model
# Everything below is model-agnostic: swap the model, keep the code.
# =============================================================
USE_MOCK = os.environ.get("OPENAI_API_KEY") is None
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")

def chat(messages, temperature=0):
    """Return the assistant's text for a list of {role, content} messages."""
    if USE_MOCK:
        return _mock(messages, temperature)
    from openai import OpenAI
    client = OpenAI(base_url=os.environ.get("OPENAI_BASE_URL"))  # None -> api.openai.com
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature, timeout=60)
    return r.choices[0].message.content

print("Backend:", "MOCK (no key found — scripted demo)" if USE_MOCK else f"REAL model = {MODEL}")

def _mock(messages, temperature=0):
    "Short canned answers so the routed 'answer' step has something to show."
    u = " ".join(m["content"] for m in messages if m["role"] == "user").lower()
    if "inr" in u:       return "INR is 4.2 (range 2.0-3.0) — above range."
    if "appointment" in u or "follow-up" in u: return "Proposed follow-up: Thursday 10:00 AM."
    if "interaction" in u or "reconcile" in u: return "Possible warfarin-amiodarone interaction; review."
    return "Done."

Backend: REAL model = openai/gpt-4o-mini


### 1) Tool retrieval — narrow the action space
A realistic agent has many tools. Exposing all of them every turn invites wrong calls and wastes context. We score each tool against the request and expose only the top few. (Here the score is simple lexical overlap; production systems use embeddings for semantic match — swap the scorer, keep the idea.)

In [2]:
TOOLS = {
 "get_latest_inr":     "retrieve the most recent INR anticoagulation lab value",
 "get_medications":    "list the patient current active medications",
 "get_recent_notes":   "read the most recent clinical progress notes",
 "get_allergies":      "list documented patient allergies",
 "get_vitals":         "retrieve latest vital signs blood pressure heart rate",
 "get_problem_list":   "list active problems and diagnoses",
 "schedule_appointment":"book a follow-up appointment visit",
 "cancel_appointment": "cancel an existing appointment",
 "send_patient_message":"send a secure message to the patient",
 "order_lab":          "place an order for a laboratory test",
 "refill_medication":  "submit a prescription medication refill",
 "calculate":          "evaluate an arithmetic expression or dose",
 "search_guidelines":  "search clinical guidelines and literature",
 "get_insurance":      "retrieve insurance coverage details",
 "submit_prior_auth":  "submit a prior authorization request",
}
STOP = set("the a an to and of for with in on is are your".split())
def toks(s): return {w for w in re.findall(r"[a-z]+", s.lower()) if w not in STOP}
def retrieve(query, k=3):
    q = toks(query)
    scored = [(len(q & toks(name+" "+desc)), name) for name, desc in TOOLS.items()]
    return [n for s, n in sorted(scored, reverse=True)[:k] if s > 0]

for query in ["get the most recent INR and the current medications",
              "schedule a follow-up appointment and message the patient"]:
    print("request:", query)
    print(f"   exposed {len(retrieve(query))} of {len(TOOLS)} tools ->", retrieve(query), "\n")

request: get the most recent INR and the current medications
   exposed 3 of 15 tools -> ['get_latest_inr', 'get_recent_notes', 'get_medications'] 

request: schedule a follow-up appointment and message the patient
   exposed 3 of 15 tools -> ['schedule_appointment', 'send_patient_message', 'get_medications'] 



### 2) Model routing — a cascade
Not every step needs the expensive model. Send simple lookups to a cheap model and reserve the strong one for multi-step reasoning. Here a keyword heuristic stands in for the router (a real one can use a small classifier, or escalate when the cheap model is unsure — see NB 6). Routing and retrieval are plain logic — like NB 7, this notebook shows plumbing, not model output.

In [3]:
COST = {"cheap": 0.001, "strong": 0.02}   # $ per call (illustrative)
HARD = ("calculate","compare","plan","synthesize","reconcile","interaction","why","diagnos","interpret")
def route(task):
    return "strong" if any(w in task.lower() for w in HARD) else "cheap"

BATCH = [
 "get the latest INR", "list current medications", "what are the documented allergies",
 "book a follow-up appointment", "reconcile meds for a warfarin drug interaction",
 "interpret the trend across the last three INR results", "send the patient a reminder",
 "plan the post-discharge follow-up steps",
]
routed = strong_only = 0.0
print(f"{'tier':>7}  task")
for t in BATCH:
    tier = route(t); routed += COST[tier]; strong_only += COST["strong"]
    print(f"{tier:>7}  {t}")          # a real system would now send this task to the chosen model
print(f"\nRouted cost:      ${routed:.3f}")
print(f"All-strong cost:  ${strong_only:.3f}")
print(f"Saved:            {100*(1-routed/strong_only):.0f}%  (same tasks, big model only where needed)")

   tier  task
  cheap  get the latest INR
  cheap  list current medications
  cheap  what are the documented allergies
  cheap  book a follow-up appointment
 strong  reconcile meds for a warfarin drug interaction
 strong  interpret the trend across the last three INR results
  cheap  send the patient a reminder
 strong  plan the post-discharge follow-up steps

Routed cost:      $0.065
All-strong cost:  $0.160
Saved:            59%  (same tasks, big model only where needed)


### Takeaway
Neither move makes the model smarter — they make the agent **cheaper, faster, and less error-prone** by shrinking what it has to choose from and reserving expensive reasoning for the steps that need it. In healthcare the retrieval step is also a *safety* lever: if a dangerous tool is never retrieved, it can't be misfired. Routing pairs naturally with the abstention idea from NB 6 — route up (or to a human) when the cheap path is unsure.

*Try:* add a tool, or change the router to escalate on low confidence instead of keywords, and watch the exposed-tool set and the cost move.